In [ ]:
!git clone https://github.com/E-tech-coder/DataScienceCapstoneProject.git

Cloning into 'DataScienceCapstoneProject'...
remote: Enumerating objects: 290, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 290 (delta 62), reused 20 (delta 20), pack-reused 212 (from 2)
Receiving objects: 100% (290/290), 2.12 MiB | 10.32 MiB/s, done.
Resolving deltas: 100% (162/162), done.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Department prediction with KNN

In [ ]:
import pandas as pd

In [ ]:
department = pd.read_csv("/content/DataScienceCapstoneProject/department.csv")
test_df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")[["position","department"]]

## Embedding

We used the sentence transformer from TechWolf/JobBERT-v3 to embed our job positions for KNN.

* This is a sentence-transformers model specifically trained for job title matching and similarity.
* It's finetuned from sentence-transformers/paraphrase-multilingual-mpnet-base-v2 on a large dataset of job titles and their associated skills/requirements across multiple languages.

In [ ]:
! pip install sentence-transformers scikit-learn

In [ ]:
# SentenceTransformer expects a Python list of strings, not a pandas Series.

X = department["text"].astype(str).tolist()
y = department["label"].astype(str).tolist()

In [ ]:
# We split the "department.csv" dataset into the training data and evaluation data.
import numpy as np
from sklearn.model_selection import train_test_split

X = department["text"]
y = department['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state= 50, test_size = 0.2, shuffle = True
)

X_train = X_train.astype(str).tolist()
X_test = X_test.astype(str).tolist()
y_train = y_train.astype(str).tolist()
y_test = y_test.astype(str).tolist()

### Create Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("TechWolf/JobBERT-v3")
X_train_embedded = model.encode(
    X_train,
    show_progress_bar = True,
    normalize_embeddings = True
)
X_test_embedded = model.encode(
    X_test,
    show_progress_bar = True,
    normalize_embeddings = True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Asym/6235903824_Dense/model.safetensor(…):   0%|          | 0.00/3.15M [00:00<?, ?B/s]

2_Asym/6235904160_Dense/model.safetensor(…):   0%|          | 0.00/3.15M [00:00<?, ?B/s]

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

# Train KNN Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(
    n_neighbors = 5,
    metric = "cosine",
    weights="distance"
)

knn.fit(X_train_embedded, y_train)

KNeighborsClassifier(metric='cosine', weights='distance')

# Evaluate

In [ ]:
for k in [3, 5, 7, 9, 11, 13, 15, 17, 19]:
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    knn.fit(X_train_embedded, y_train)
    score = knn.score(X_test_embedded, y_test)
    print(f"K={k}, Accuracy={score:.3f}")

K=3, Accuracy=0.930
K=5, Accuracy=0.928
K=7, Accuracy=0.931
K=9, Accuracy=0.932
K=11, Accuracy=0.932
K=13, Accuracy=0.931
K=15, Accuracy=0.933
K=17, Accuracy=0.931
K=19, Accuracy=0.929


In [ ]:
knn = KNeighborsClassifier(
    n_neighbors = 15,
    metric = "cosine",
    weights="distance"
)
knn.fit(X_train_embedded, y_train)
from sklearn.metrics import classification_report
y_pred = knn.predict(X_test_embedded)
print(classification_report(y_pred, y_test))

                        precision    recall  f1-score   support

        Administrative       0.83      0.94      0.88        16
  Business Development       0.78      0.92      0.84        84
            Consulting       0.64      0.68      0.66        31
      Customer Support       0.14      1.00      0.25         1
       Human Resources       0.25      0.50      0.33         2
Information Technology       0.94      0.85      0.89       278
             Marketing       0.98      0.95      0.96       918
                 Other       0.67      1.00      0.80         6
    Project Management       0.45      0.72      0.55        18
            Purchasing       1.00      0.83      0.91         6
                 Sales       0.96      0.96      0.96       669

              accuracy                           0.93      2029
             macro avg       0.69      0.85      0.73      2029
          weighted avg       0.94      0.93      0.94      2029



### we have the highest accuracy when k = 15 , so we will use k = 15 in the following evalution process. On the test dataset from the "department.csv", we got an accuracy of 93.3%

In [ ]:
k = 15
knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
knn.fit(X_train_embedded, y_train)

KNeighborsClassifier(metric='cosine', n_neighbors=15)

## Measure the accuracy on the CV dataset

In [ ]:
eval_df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
X_eval = eval_df["position"].astype(str).tolist()
y_eval = eval_df["department"].astype(str).tolist()

X_eval_embedded = model.encode(
    X_eval,
    show_progress_bar = True,
    normalize_embeddings = True
)


Batches:   0%|          | 0/82 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics import classification_report
y_pred = knn.predict(X_eval_embedded)
print(classification_report(y_eval, y_pred))

                        precision    recall  f1-score   support

        Administrative       0.30      0.39      0.34        84
  Business Development       0.25      0.82      0.38        78
            Consulting       0.75      0.50      0.60       195
      Customer Support       0.88      0.15      0.25        48
       Human Resources       0.82      0.68      0.75        69
Information Technology       0.65      0.67      0.66       309
             Marketing       0.15      0.88      0.26       133
                 Other       0.94      0.05      0.10      1235
    Project Management       0.49      0.58      0.53       173
            Purchasing       0.76      0.74      0.75        72
                 Sales       0.31      0.89      0.46       219

              accuracy                           0.38      2615
             macro avg       0.57      0.58      0.46      2615
          weighted avg       0.72      0.38      0.33      2615



In the CV dataset, our prediction accuracy dropped to only 37%.
One of the main reasons is label disrtribution shift.
In our training data, the KNN model didn't get enough semantic training on the "Other" department, while in the test dataset half of the data is labeled as "Other".


## Measure the accuracy on the CV dataset without "Other" department

In [ ]:
eval_df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
eval_df = eval_df[eval_df["department"]!= "Other"]
X_eval = eval_df["position"].astype(str).tolist()
y_eval = eval_df["department"].astype(str).tolist()

X_eval_embedded = model.encode(
    X_eval,
    show_progress_bar = True,
    normalize_embeddings = True
)

from sklearn.metrics import classification_report
y_pred = knn.predict(X_eval_embedded)
print(classification_report(y_eval, y_pred))


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

                        precision    recall  f1-score   support

        Administrative       0.83      0.42      0.56        84
  Business Development       0.49      0.83      0.62        78
            Consulting       0.89      0.48      0.62       195
      Customer Support       1.00      0.15      0.25        48
       Human Resources       0.92      0.70      0.79        69
Information Technology       0.80      0.66      0.72       309
             Marketing       0.44      0.88      0.59       133
                 Other       0.00      0.00      0.00         0
    Project Management       0.74      0.58      0.65       173
            Purchasing       0.98      0.72      0.83        72
                 Sales       0.59      0.88      0.71       219

              accuracy                           0.66      1380
             macro avg       0.70      0.57      0.58      1380
          weighted avg       0.74      0.66      0.66      1380



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# KNN with distance threshold

In [ ]:
# Fit KNN normally
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(
    n_neighbors=15,
    metric="cosine",
    weights="distance"
)
knn.fit(X_train_embedded, y_train)

# Calculate the distances with the Linkedin data
eval_df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
X_eval = eval_df["position"].astype(str).tolist()
y_eval = eval_df["department"].astype(str).tolist()

# Evaluate with the linkedinCV data
X_eval_embedded = model.encode(
    X_eval,
    show_progress_bar = True,
    normalize_embeddings = True
)

distances, indices = knn.kneighbors(X_eval_embedded)
# distances: shape (n_samples, k)

Batches:   0%|          | 0/82 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics import accuracy_score
max_sim = 1- distances.min(axis = 1)
threshold = np.linspace(0.5,1.0,20)
# Calculate the smallest distance of our evalution sample to the nearest 15 training samples, to see if this evaluation sample is far from our training data.
# If the evaluation sample is too far from its nearest 15 training data neighbours,exceeding the threshold. We ditch it to "Other" department.
for t in threshold :
  is_other = max_sim < t
  y_pred = knn.predict(X_eval_embedded)
  y_pred_adj = y_pred.copy()
  y_pred_adj[is_other] = "Other"
  acc = accuracy_score(y_pred_adj, y_eval)
  print(f"Threshold ={t :.3f}, Accuracy={acc:.3f}")


Threshold =0.500, Accuracy=0.478
Threshold =0.526, Accuracy=0.504
Threshold =0.553, Accuracy=0.524
Threshold =0.579, Accuracy=0.540
Threshold =0.605, Accuracy=0.550
Threshold =0.632, Accuracy=0.564
Threshold =0.658, Accuracy=0.571
Threshold =0.684, Accuracy=0.574
Threshold =0.711, Accuracy=0.580
Threshold =0.737, Accuracy=0.590
Threshold =0.763, Accuracy=0.603
Threshold =0.789, Accuracy=0.631
Threshold =0.816, Accuracy=0.633
Threshold =0.842, Accuracy=0.628
Threshold =0.868, Accuracy=0.617
Threshold =0.895, Accuracy=0.608
Threshold =0.921, Accuracy=0.594
Threshold =0.947, Accuracy=0.581
Threshold =0.974, Accuracy=0.571
Threshold =1.000, Accuracy=0.511


In [ ]:
from sklearn.metrics import accuracy_score

max_sim = 1- distances.min(axis = 1)
threshold = 0.816
is_other = max_sim < threshold
y_pred = knn.predict(X_eval_embedded)
y_pred_adj = y_pred.copy()
y_pred_adj[is_other] = "Other"

print(classification_report(y_pred_adj, y_eval))

                        precision    recall  f1-score   support

        Administrative       0.18      0.88      0.30        17
  Business Development       0.44      0.49      0.46        70
            Consulting       0.34      0.99      0.50        67
      Customer Support       0.04      0.67      0.08         3
       Human Resources       0.43      0.94      0.59        32
Information Technology       0.28      0.82      0.42       105
             Marketing       0.47      0.51      0.49       123
                 Other       0.93      0.59      0.72      1962
    Project Management       0.35      0.95      0.51        63
            Purchasing       0.32      0.85      0.46        27
                 Sales       0.56      0.84      0.67       146

              accuracy                           0.63      2615
             macro avg       0.39      0.77      0.47      2615
          weighted avg       0.80      0.63      0.67      2615



# Accuracy = 63.3%
With our distance threshold method, if the sample is too far from our training data, we dump it to the "Other" label. It gives us a highest accuracy of 63.3%, with a similarity threshold of 0.816.